# Week 1: Python & NumPy Foundations
## Notebook 06: Einstein Summation (np.einsum) & Tensor Contractions from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In Deep Learning and scientific computing, matrix operations frequently exceed 2 dimensions:
- Attention layers compute batched multi-head dot products: `[batch, heads, seq_len, head_dim]`.
- Computer vision processes 4D tensors: `[batch, channels, height, width]`.
- Graph neural networks compute message aggregations across adjacency matrices and feature nodes.

Using chained combinations of `transpose()`, `reshape()`, and `matmul()` is notoriously error-prone, hard to read, and often creates unnecessary intermediate array copies in memory.

**Einstein Summation (`np.einsum`)** provides a universal, concise, and memory-efficient syntax for any tensor contraction. By specifying indices, NumPy automatically derives loops, sums, and axis permutations under optimized C loops!

---

### 2. Mathematical Foundations: Einstein Notation
In classical physics, Albert Einstein proposed that whenever an index variable appears twice in a single term and is not otherwise defined, it implies summation of that term over all possible values of the index:

$$c_{ik} = \sum_j a_{ij} b_{jk} \quad \implies \quad c_{ik} = a_{ij} b_{jk}$$

In `np.einsum(subscripts, *operands)`:
1. **Matrix Multiplication ($A B$):** `'ij,jk->ik'`
   - Index `j` appears in both inputs but NOT in output $\implies$ Sum over `j`.
2. **Matrix Trace ($\text{Tr}(A) = \sum_i A_{ii}$):** `'ii->'`
   - Diagonal elements summed to a scalar.
3. **Batched Matrix Multiplication:** `'bij,bjk->bik'`
   - Preserves batch dimension `b` while multiplying `N` matrices in parallel.
4. **Multi-Head Attention Attention-Score Matrix ($Q K^T$):**
   $$S_{b, h, i, j} = \sum_{d} Q_{b, h, i, d} K_{b, h, j, d} \quad \implies \quad \text{'bhid,bhjd->bhij'}$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Classical 2D Operations using einsum
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# 1. Matrix Multiplication (ij, jk -> ik)
matmul_einsum = np.einsum('ij,jk->ik', A, B)
matmul_standard = A @ B
print("--- 1. Matrix Multiplication ---")
print("Einsum:\n", matmul_einsum)
print("Standard Matches:", np.allclose(matmul_einsum, matmul_standard))

# 2. Matrix Trace (ii -> scalar)
trace_einsum = np.einsum('ii->', A)
print("\n--- 2. Matrix Trace ---")
print(f"Trace Einsum: {trace_einsum} | np.trace: {np.trace(A)}")

# 3. Outer Product of Vectors (i, j -> ij)
u = np.array([1, 2, 3])
v = np.array([4, 5])
outer_einsum = np.einsum('i,j->ij', u, v)
print("\n--- 3. Outer Product (3x1 and 1x2 -> 3x2) ---")
print(outer_einsum)

# Step 2: High-Dimensional Batched Multi-Head Attention Contraction
B_size = 2   # Batch size
H_heads = 4  # Number of attention heads
S_len = 8    # Sequence length
D_dim = 16   # Head dimension

# Query (Q) and Key (K) tensors: [Batch, Heads, Seq_len, Head_dim]
Q = np.random.randn(B_size, H_heads, S_len, D_dim)
K = np.random.randn(B_size, H_heads, S_len, D_dim)

# Target: Attention Weights Score Matrix S = Q @ K.T (batched)
# Shape: [Batch, Heads, Seq_len, Seq_len]
# Formula: S[b, h, i, j] = sum_d (Q[b, h, i, d] * K[b, h, j, d])
scores_einsum = np.einsum('bhid,bhjd->bhij', Q, K)

# Verify with standard transpose + matmul
scores_standard = np.matmul(Q, K.transpose(0, 1, 3, 2))
print("\n--- 4. Batched Multi-Head Attention Tensor Contraction ---")
print("Scores Shape:", scores_einsum.shape)
print("Exact numerical match with matmul:", np.allclose(scores_einsum, scores_standard))

### 3. Step-by-Step Code Walkthrough
1. **Index Specification:** In `'ij,jk->ik'`, `i` is row of A, `j` is column of A / row of B, and `k` is column of B.
2. **Elimination via Summation:** Since `j` is omitted after `->`, NumPy accumulates products across all values of `j`.
3. **Batched Attention:** In `'bhid,bhjd->bhij'`, `d` is the feature dimension being contracted, while `b`, `h`, `i`, and `j` are preserved, generating an $[8 \times 8]$ attention grid for every head in every batch item.

---

### 4. Industry Applications & Trade-offs
- **Transformers (PyTorch & JAX):** `torch.einsum` is widely used in Hugging Face Transformers and RoPE implementations for self-documenting tensor algebra.
- **Trade-off:** For standard 2D matrix multiplication, `A @ B` directly invokes vendor-tuned BLAS (GEMM) which can be slightly faster. For 3D+ contractions, `optimize=True` in `np.einsum` finds optimal contraction paths.

---

### 5. Top Interview Questions & Answers
**Q1: How does `optimize=True` work in `np.einsum`?**
> **Answer:** When multiplying 3 or more tensors (e.g. $A B C$), matrix multiplication is associative: $(A B) C = A (B C)$. However, the computational complexity drastically differs depending on intermediate dimension sizes. `optimize=True` uses dynamic programming to find the contraction sequence with minimum floating-point operations (FLOPs).

**Q2: What happens if you omit the arrow `->` in an einsum subscript?**
> **Answer:** If the output subscript is omitted (e.g., `np.einsum('ij,jk', A, B)`), NumPy defaults to Einstein's implicit rule: any index appearing once in the inputs is preserved in alphabetical order, and indices appearing multiple times are summed.